### Reference

This project's methodology (ticker selection, weight-generation scheme, volatility target construction, and RF-based risk classification/regression) follows and extends:

> Rathi, V., Kshirsagar, M., & Ryan, C. (2024). Enhancing Portfolio Performance: A Random Forest Approach to Volatility Prediction and Optimization. *Proceedings of the 16th International Conference on Agents and Artificial Intelligence (ICAART 2024)*, 1278–1285. https://doi.org/10.5220/0012464600003636

Extensions made in this project: XGBoost as a second model family, Optuna-based hyperparameter tuning (vs. grid search in the original), SHAP-based interpretability analysis, and modern portfolio theory optimization via `scipy.optimize`.

# Importing Libraries

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import joblib

In [2]:
tickers = [
    "VTSAX",
    "VTIAX",
    "VBTLX",
    "VDE",
    "VGSLX",
    "OPGSX",
    "BTC-USD"
]

### Data Collection

In [3]:
data = yf.download(
    tickers,
    start="2015-01-01",
    end="2025-12-31",
    auto_adjust=True
)

[*********************100%***********************]  7 of 7 completed


Historical price data for the selected portfolio assets were collected using the **Yahoo Finance (yfinance)** Python library. Daily adjusted closing prices were downloaded for the period from **January 1, 2015, to December 31, 2025** using the `download()` function with `auto_adjust=True`, which automatically accounts for stock splits and dividend adjustments. The collected data served as the basis for calculating daily returns, estimating expected returns, computing the covariance matrix, and performing portfolio optimization and machine learning analysis.

In [4]:
joblib.dump(data,"data.pkl")

['data.pkl']

In [5]:
data.head(10)


Price            Close                                                        \
Ticker         BTC-USD      OPGSX     VBTLX        VDE      VGSLX      VTIAX   
Date                                                                           
2015-01-01  314.248993        NaN       NaN        NaN        NaN        NaN   
2015-01-02  315.032013  11.681229  7.895160  75.751503  74.034370  18.581963   
2015-01-03  281.082001        NaN       NaN        NaN        NaN        NaN   
2015-01-04  264.195007        NaN       NaN        NaN        NaN        NaN   
2015-01-05  274.473999  11.973678  7.916890  72.746567  74.434616  18.202148   
2015-01-06  286.188995  12.566933  7.945862  71.652626  75.203278  18.030161   
2015-01-07  294.337006  12.349684  7.945862  71.828217  76.353134  18.173485   
2015-01-08  283.348999  12.107369  7.924134  73.381325  76.588173  18.460136   
2015-01-09  290.407990  12.533509  7.945862  72.868103  76.645348  18.359808   
2015-01-10  274.795990        NaN       NaN        NaN        NaN        NaN   

Price                        High                       ...       Open  \
Ticker          VTSAX     BTC-USD      OPGSX     VBTLX  ...      VGSLX   
Date                                                    ...              
2015-01-01        NaN  320.434998        NaN       NaN  ...        NaN   
2015-01-02  42.690781  315.838989  11.681229  7.895160  ...  74.034370   
2015-01-03        NaN  315.149994        NaN       NaN  ...        NaN   
2015-01-04        NaN  287.230011        NaN       NaN  ...        NaN   
2015-01-05  41.937458  278.341003  11.973678  7.916890  ...  74.434616   
2015-01-06  41.531822  287.553009  12.566933  7.945862  ...  75.203278   
2015-01-07  42.028515  298.753998  12.349684  7.945862  ...  76.353134   
2015-01-08  42.765278  294.135010  12.107369  7.924134  ...  76.588173   
2015-01-09  42.417606  291.114014  12.533509  7.945862  ...  76.645348   
2015-01-10        NaN  288.127014        NaN       NaN  ...        NaN   

Price                               Volume                                     \
Ticker          VTIAX      VTSAX   BTC-USD OPGSX VBTLX        VDE VGSLX VTIAX   
Date                                                                            
2015-01-01        NaN        NaN   8036550   NaN   NaN        NaN   NaN   NaN   
2015-01-02  18.581963  42.690781   7860650   0.0   0.0  1694200.0   0.0   0.0   
2015-01-03        NaN        NaN  33054400   NaN   NaN        NaN   NaN   NaN   
2015-01-04        NaN        NaN  55629100   NaN   NaN        NaN   NaN   NaN   
2015-01-05  18.202148  41.937458  43962800   0.0   0.0  1016500.0   0.0   0.0   
2015-01-06  18.030161  41.531822  23245700   0.0   0.0   957800.0   0.0   0.0   
2015-01-07  18.173485  42.028515  24866800   0.0   0.0   710700.0   0.0   0.0   
2015-01-08  18.460136  42.765278  19982500   0.0   0.0   761900.0   0.0   0.0   
2015-01-09  18.359808  42.417606  18718600   0.0   0.0   683100.0   0.0   0.0   
2015-01-10        NaN        NaN  15264300   NaN   NaN        NaN   NaN   NaN   

Price             
Ticker     VTSAX  
Date              
2015-01-01   NaN  
2015-01-02   0.0  
2015-01-03   NaN  
2015-01-04   NaN  
2015-01-05   0.0  
2015-01-06   0.0  
2015-01-07   0.0  
2015-01-08   0.0  
2015-01-09   0.0  
2015-01-10   NaN  

[10 rows x 35 columns]

In [6]:
close_Price=data["Close"]
close_Price.head()

Ticker,BTC-USD,OPGSX,VBTLX,VDE,VGSLX,VTIAX,VTSAX
Date,,,,,,,
2015-01-01,314.248993,NaN,NaN,NaN,NaN,NaN,NaN
2015-01-02,315.032013,11.681229,7.89516,75.751503,74.034370,18.581963,42.690781
2015-01-03,281.082001,NaN,NaN,NaN,NaN,NaN,NaN
2015-01-04,264.195007,NaN,NaN,NaN,NaN,NaN,NaN
2015-01-05,274.473999,11.973678,7.91689,72.746567,74.434616,18.202148,41.937458


### Data Cleaning

In [7]:
close_Price=close_Price.dropna()
close_Price.head()

Ticker,BTC-USD,OPGSX,VBTLX,VDE,VGSLX,VTIAX,VTSAX
Date,,,,,,,
2015-01-02,315.032013,11.681229,7.895160,75.751503,74.034370,18.581963,42.690781
2015-01-05,274.473999,11.973678,7.916890,72.746567,74.434616,18.202148,41.937458
2015-01-06,286.188995,12.566933,7.945862,71.652626,75.203278,18.030161,41.531822
2015-01-07,294.337006,12.349684,7.945862,71.828217,76.353134,18.173485,42.028515
2015-01-08,283.348999,12.107369,7.924134,73.381325,76.588173,18.460136,42.765278


The downloaded closing price data were cleaned by removing all rows containing missing values using the `dropna()` function. This ensured that each trading day had complete price information for all selected assets, preventing errors during the calculation of daily returns, expected returns, covariance matrices, and portfolio optimization. The cleaned dataset provided a consistent and reliable basis for subsequent financial analysis.

### Daily Return Calculation

Daily returns were computed using the `pct_change()` function, which calculates the percentage change in closing prices between consecutive trading days. The first row, which contains missing values due to the absence of a previous trading day, was removed using `dropna()`. The resulting daily return dataset forms the basis for estimating expected returns, calculating the covariance matrix, measuring portfolio risk, and performing portfolio optimization.

In [8]:
returns=close_Price.pct_change().dropna()
returns.head()

Ticker,BTC-USD,OPGSX,VBTLX,VDE,VGSLX,VTIAX,VTSAX
Date,,,,,,,
2015-01-05,-0.128743,0.025036,0.002752,-0.039668,0.005406,-0.020440,-0.017646
2015-01-06,0.042682,0.049547,0.003659,-0.015038,0.010327,-0.009449,-0.009672
2015-01-07,0.028471,-0.017287,0.000000,0.002451,0.015290,0.007949,0.011959
2015-01-08,-0.037331,-0.019621,-0.002734,0.021623,0.003078,0.015773,0.017530
2015-01-09,0.024913,0.035197,0.002742,-0.006994,0.000747,-0.005435,-0.008130


In [9]:
returns.columns

Index(['BTC-USD', 'OPGSX', 'VBTLX', 'VDE', 'VGSLX', 'VTIAX', 'VTSAX'], dtype='object', name='Ticker')

In [10]:
print(close_Price.shape)
print(returns.shape)
print(returns.isnull().sum())

(2765, 7)
(2764, 7)
Ticker
BTC-USD    0
OPGSX      0
VBTLX      0
VDE        0
VGSLX      0
VTIAX      0
VTSAX      0
dtype: int64


### Random Portfolio Generation

Random portfolio allocations were generated using the **Dirichlet distribution**, which produces weight vectors that are non-negative and sum to one, ensuring fully invested portfolios. Additional allocation constraints were imposed to limit the maximum investment in selected assets: **BTC-USD** to 10%, **OPGSX** to 20%, and **VGSLX** to 20%. Portfolios satisfying these constraints were retained for further analysis, resulting in realistic and diversified asset allocations for portfolio optimization and machine learning.

In [11]:


def generate_portfolio():

    while True:

        w = np.random.dirichlet(np.ones(7))

        if (
            w[0] <= 0.10 and   # BTC
            w[1] <= 0.20 and   # OPGSX
            w[4] <= 0.20       # VGSLX
        ):
            return w

In [12]:
weights = generate_portfolio()

print(weights)
print(weights.sum())

[0.02753562 0.01505467 0.68690893 0.10092441 0.02555018 0.02371433
 0.12031187]
1.0


### Portfolio Generation

A total of **10,000 random portfolios** were generated by repeatedly calling the `generate_portfolio()` function. Each portfolio consists of asset weights that satisfy the predefined allocation constraints while ensuring that the total investment sums to 100%. The generated portfolios were stored in a list and then converted into a NumPy array to enable efficient numerical computations for portfolio return estimation, risk analysis, and machine learning model development.

In [13]:
portfolios = []

for i in range(10000):

    portfolios.append(generate_portfolio())

portfolios = np.array(portfolios)

In [14]:
print(portfolios.shape)

(10000, 7)


In [15]:
portfolio_data = pd.DataFrame(
    portfolios,
    columns=returns.columns
)

portfolio_data.head()

Ticker,BTC-USD,OPGSX,VBTLX,VDE,VGSLX,VTIAX,VTSAX
0,0.038260,0.094371,0.068007,0.484369,0.042025,0.122006,0.150963
1,0.094819,0.076806,0.058109,0.093447,0.161409,0.361310,0.154100
2,0.090654,0.172981,0.278093,0.163983,0.031532,0.212984,0.049772
3,0.025827,0.015625,0.001488,0.418204,0.031454,0.057249,0.450154
4,0.047786,0.187009,0.055847,0.027544,0.042493,0.321089,0.318233


In [16]:
portfolio_data.shape

(10000, 7)

### Portfolio Volatility Calculation

The annualized volatility of each generated portfolio was calculated using the weighted daily returns of the constituent assets. For each portfolio, daily portfolio returns were obtained by multiplying the asset return matrix with the corresponding portfolio weights. The standard deviation of these daily returns was then annualized by multiplying by the square root of **252**, representing the average number of trading days in a year. The computed volatility values were stored as a new **Volatility** column in the portfolio dataset, providing a quantitative measure of portfolio risk for subsequent analysis and machine learning.

In [17]:
volatility = []

for _, weights in portfolio_data.iterrows():

    daily_portfolio_return = returns.dot(weights.values)

    annual_vol = daily_portfolio_return.std() * np.sqrt(252)

    volatility.append(annual_vol)

portfolio_data["Volatility"] = volatility

In [18]:
print(portfolio_data.head())
print("SHAPE OF THE DATA : ",portfolio_data.shape)


Ticker   BTC-USD     OPGSX     VBTLX       VDE     VGSLX     VTIAX     VTSAX  \
0       0.038260  0.094371  0.068007  0.484369  0.042025  0.122006  0.150963   
1       0.094819  0.076806  0.058109  0.093447  0.161409  0.361310  0.154100   
2       0.090654  0.172981  0.278093  0.163983  0.031532  0.212984  0.049772   
3       0.025827  0.015625  0.001488  0.418204  0.031454  0.057249  0.450154   
4       0.047786  0.187009  0.055847  0.027544  0.042493  0.321089  0.318233   

Ticker  Volatility  
0         0.201017  
1         0.166520  
2         0.147172  
3         0.204879  
4         0.161963  
SHAPE OF THE DATA :  (10000, 8)


### Risk Category Assignment

The generated portfolios were classified into four risk categories based on their annualized volatility using the `pd.qcut()` function. Volatility values were divided into four equal-sized groups (quartiles) and labeled as **Low**, **Medium**, **Moderate**, and **High**. This approach ensured a balanced distribution of portfolios across the risk categories, making the dataset suitable for training and evaluating machine learning classification models.

In [19]:
portfolio_data["Risk_category"] = pd.qcut(
    portfolio_data["Volatility"],
    q=4,
    labels=["Low", "Medium", "Moderate", "High"]
)

In [20]:
portfolio_data.head()

Ticker,BTC-USD,OPGSX,VBTLX,VDE,VGSLX,VTIAX,VTSAX,Volatility,Risk_category
0,0.038260,0.094371,0.068007,0.484369,0.042025,0.122006,0.150963,0.201017,High
1,0.094819,0.076806,0.058109,0.093447,0.161409,0.361310,0.154100,0.166520,Moderate
2,0.090654,0.172981,0.278093,0.163983,0.031532,0.212984,0.049772,0.147172,Medium
3,0.025827,0.015625,0.001488,0.418204,0.031454,0.057249,0.450154,0.204879,High
4,0.047786,0.187009,0.055847,0.027544,0.042493,0.321089,0.318233,0.161963,Moderate


In [21]:
portfolio_data["Risk_category"].value_counts()

Risk_category
Low         2500
Medium      2500
Moderate    2500
High        2500
Name: count, dtype: int64

In [22]:
print(portfolio_data["Volatility"].min())
print(portfolio_data["Volatility"].max())
portfolio_data["Volatility"].describe()

0.04955612349980458
0.2715240959051926


count    10000.000000
mean         0.153526
std          0.030722
min          0.049556
25%          0.133655
50%          0.155582
75%          0.172621
max          0.271524
Name: Volatility, dtype: float64

In [23]:
print(returns.mean())
print(returns.std())

Ticker
BTC-USD    0.002933
OPGSX      0.000801
VBTLX      0.000077
VDE        0.000359
VGSLX      0.000272
VTIAX      0.000331
VTSAX      0.000552
dtype: float64
Ticker
BTC-USD    0.042102
OPGSX      0.020523
VBTLX      0.003069
VDE        0.018775
VGSLX      0.013012
VTIAX      0.009976
VTSAX      0.011479
dtype: float64


In [24]:
portfolio_data.head(10)

Ticker,BTC-USD,OPGSX,VBTLX,VDE,VGSLX,VTIAX,VTSAX,Volatility,Risk_category
0,0.038260,0.094371,0.068007,0.484369,0.042025,0.122006,0.150963,0.201017,High
1,0.094819,0.076806,0.058109,0.093447,0.161409,0.361310,0.154100,0.166520,Moderate
2,0.090654,0.172981,0.278093,0.163983,0.031532,0.212984,0.049772,0.147172,Medium
3,0.025827,0.015625,0.001488,0.418204,0.031454,0.057249,0.450154,0.204879,High
4,0.047786,0.187009,0.055847,0.027544,0.042493,0.321089,0.318233,0.161963,Moderate
5,0.064159,0.110744,0.349386,0.210515,0.087614,0.046937,0.130644,0.132720,Low
6,0.034638,0.014539,0.365279,0.233074,0.047590,0.075835,0.229045,0.125831,Low
7,0.092200,0.107943,0.152676,0.157070,0.090589,0.182488,0.217034,0.159774,Moderate
8,0.019500,0.011579,0.425594,0.192386,0.012843,0.154998,0.183101,0.109874,Low
9,0.064338,0.184830,0.127688,0.009447,0.087931,0.280574,0.245192,0.153427,Medium


In [25]:
import joblib
joblib.dump(returns,"Returns.pkl")
joblib.dump(portfolio_data,"portfolio_data.pkl")

['portfolio_data.pkl']